In [1]:
import torch
from dataset.climatenet import ClimateDataset 
from torch.utils.data import DataLoader
from functools import partial
import random
import numpy as np
import os
from train_util import batch_to_cuda, get_idle_gpu, get_idle_port, set_randomness,  plot_with_projection, plot_mask_with_points_and_bbox, prompt_debug, setup_device_and_distributed, setup_optimizer_and_scheduler, worker_init_fn

def set_up_dataset(worker_args):
    dataset_dir = worker_args.data_dir
    train_bs = worker_args.train_bs 
    val_bs = worker_args.val_bs
    gradient_accumulation_steps = worker_args.gradient_accumulation_steps
    
    
    train_dataset = ClimateDataset(
        data_dir=dataset_dir, train_flag=True, shot_num=worker_args.shot_num,
        augmented=worker_args.augmented, generate_prompt=True
    )
    val_dataset = ClimateDataset(data_dir=dataset_dir, train_flag=False, augmented=False, generate_prompt=True)
    train_collate_fn = train_dataset.collate_fn
    val_collate_fn = val_dataset.collate_fn
    # Debugging mode - use smaller dataset and fewer epochs
    if hasattr(worker_args, 'debugging') and worker_args.debugging:
        debug_size = getattr(worker_args, 'debug_size', 10)  # Default to 50 samples
        indices = list(range(min(debug_size, len(train_dataset))))
        train_dataset = torch.utils.data.Subset(train_dataset, indices)
        print(f"Debug mode: Using only {len(train_dataset)} training samples")

        debug_val_size = getattr(worker_args, 'debug_val_size', 5)  # Default to 10 samples
        val_indices = list(range(min(debug_val_size, len(val_dataset))))
        val_dataset = torch.utils.data.Subset(val_dataset, val_indices)
        print(f"Debug mode: Using only {len(val_dataset)} validation samples")
        
        max_epoch_num = 2
        worker_args.valid_per_epochs = 1
        print(f"Debug mode: Setting max_epoch_num to {max_epoch_num} and valid_per_epochs to {worker_args.valid_per_epochs}")
        
    # Adjust batch size for gradient accumulation
    actual_train_bs = train_bs // gradient_accumulation_steps
    if actual_train_bs < 1:
        actual_train_bs = 1
        print(f"Warning: gradient_accumulation_steps ({gradient_accumulation_steps}) is larger than train_bs ({train_bs}). Setting actual batch size to 1.")
    
    effective_batch_size = actual_train_bs * gradient_accumulation_steps
    
    print(f"Effective batch size: {effective_batch_size} (actual_bs: {actual_train_bs}, accumulation: {gradient_accumulation_steps})")
        
    train_workers, val_workers = 4, 2
    
    sampler = None
        
    train_dataloader = DataLoader(
        dataset=train_dataset, batch_size=actual_train_bs, shuffle=sampler is None, num_workers=train_workers,
        sampler=sampler, drop_last=False, collate_fn=train_collate_fn,
        worker_init_fn=partial(worker_init_fn, base_seed=3407)
    )
    val_dataloader = DataLoader(
        dataset=val_dataset, batch_size=val_bs, shuffle=False, num_workers=val_workers,
        drop_last=False, collate_fn=val_collate_fn, worker_init_fn=partial(worker_init_fn, base_seed=3407)
    )
    
    return train_dataloader, val_dataloader

In [2]:
from dataset.climatenet import ClimateDataset
from model.prompt_generator import PromptGenerator
from model.prompt.prompt_maker import PromptMaker
from climatesam import ClimateSAM

def set_up_model(worker_args, device):
    climatesam = ClimateSAM(
        model_type=worker_args.sam_type, 
        mlp_ratio=worker_args.image_encoder_mlp_ratio,
        enable_wandb_logging=getattr(worker_args, 'debugging', False)  # Only log if debugging=True
    ).to(device)
    
    # Load pretrained weights
    if worker_args.load_pretrained:
        if worker_args.phase == 1:
            image_encoder_path = os.path.join(worker_args.exp_dir,'best_weights', f"phase_2_weights_best.pth")
            phase_1_checkpoint = torch.load(image_encoder_path, map_location=device)
            print(f"Pretrained weights from phase 1 loaded from {image_encoder_path}")
            climatesam.image_encoder.load_state_dict(phase_1_checkpoint['image_encoder'])
            print(f"Image encoder weights loaded from {image_encoder_path}")
            climatesam.mask_decoder.load_state_dict(phase_1_checkpoint['mask_decoder'])
            print(f"Mask decoder weights loaded from {image_encoder_path}")
    
    
    ###################################################
    num_features_map = {
        'vit_b': 12,
        'vit_l': 24,
        'vit_h': 32
    }
    features_per_block = {
        'vit_b': 3,
        'vit_l': 6,
        'vit_h': 9
    }
    
    in_channels = {
        'vit_b': 768,
        'vit_l': 1024,
        'vit_h': 1280
    }
    
    prompt_generator = PromptGenerator(
        in_channels=in_channels[worker_args.sam_type],
        fused_channels=1,
        num_features=num_features_map[worker_args.sam_type],
        features_per_block=features_per_block[worker_args.sam_type]
    ).to(device)
    
    for params in climatesam.image_encoder.parameters():
        params.requires_grad = False
    for params in climatesam.mask_decoder.parameters():
        params.requires_grad = False
        

    return climatesam, prompt_generator

In [3]:
from train_parser import parse
import sys
original_argv = sys.argv
sys.argv = ['train_complete_test.ipynb', '--config', 'input_config_test']
worker_args = parse()
max_epoch_num = worker_args.max_epoch_num 
train_dataloader, val_dataloader = set_up_dataset(worker_args)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
climatesam, prompt_generator = set_up_model(worker_args, device)


Debug mode: Using only 10 training samples
Debug mode: Using only 5 validation samples
Debug mode: Setting max_epoch_num to 2 and valid_per_epochs to 1
Effective batch size: 2 (actual_bs: 2, accumulation: 1)


In [4]:
prompt_maker = PromptMaker(prompt_type='point', positive_point_num=worker_args.positive_point_num, negative_point_num=worker_args.negative_point_num)

In [7]:
import torch.nn.functional as F
from loss_function import ClimateLoss, compute_climate_loss, compute_generator_loss
scaler = torch.amp.GradScaler('cuda') 
from contextlib import nullcontext

for train_step, batch in enumerate(train_dataloader):
    batch = batch_to_cuda(batch, device)
    empty_prompt_flag = False
    with torch.amp.autocast('cuda'):
            # Step 1: Encode images to get intermediate features
            image_embeddings, interm_features, image_input, ori_img_size = climatesam.encode_images(batch['input'])
            
            # Step 2: Generate masks and auxiliary predictions from intermediate features
            final_logit, interm_masks = prompt_generator(interm_features)
            
             # Use Softmax to get probability maps for visualization
            multiclass_mask = F.softmax(final_logit, dim=1)
            
            # Step 3: Create prompts from generated masks
            prompt_dict = prompt_maker.make_prompts(multiclass_mask)
            
            # if prompt_dict has None prompts, skip this batch
            
        #     for i in range(len(prompt_dict['ar_point_prompts'])):
        #         if (prompt_dict['ar_point_prompts'][i] is None and prompt_dict['tc_point_prompts'][i] is None and
        #                         prompt_dict['ar_bbox_prompts'][i] is None and prompt_dict['tc_bbox_prompts'][i] is None or
        #                         prompt_dict['ar_mask_prompts'][i] is None or prompt_dict['tc_mask_prompts'][i] is None):
        #                         print("Skipping batch due to missing prompts")
        #                         empty_prompt_flag = True
        #                         break
            
        #     if not empty_prompt_flag:
            tc_pred_masks, ar_pred_masks, _ = climatesam.forward(
                        image_input=image_input,
                        image_embeddings=image_embeddings,
                        interm_embeddings=interm_features,
                        ori_img_size=ori_img_size,
                        ar_point_prompts=prompt_dict['ar_point_prompts'],
                        tc_point_prompts=prompt_dict['tc_point_prompts'],
                        ar_bbox_prompts=prompt_dict['ar_bbox_prompts'],
                        tc_bbox_prompts=prompt_dict['tc_bbox_prompts'],
                        ar_mask_prompts=prompt_dict['ar_mask_prompts'],
                        tc_mask_prompts=prompt_dict['tc_mask_prompts']
                )
            
            # Ground truth masks
            gt_masks = torch.stack(batch['gt_mask'], dim=0).to(device)  # B, H, W
            masks_ar_gt = prompt_dict['ar_object_masks']
            masks_tc_gt = prompt_dict['tc_object_masks']
            
            
            # Compute model loss (final predictions)
            loss_model = compute_climate_loss(
                ar_masks=ar_pred_masks,
                tc_masks=tc_pred_masks,
                ar_masks_gt=masks_ar_gt,
                tc_masks_gt=masks_tc_gt,
                device=device,
                worker_args=worker_args
            )
            
            # Compute generator loss (auxiliary predictions)
            loss_gen = compute_generator_loss(
                multiclass_mask=multiclass_mask,
                interm_masks=interm_masks,
                gt_masks=gt_masks,
                device=device,
                worker_args=worker_args
            )
            
            # Combine losses
            loss_dict = {}
            loss_dict.update({f"gen_{k}": v for k, v in loss_gen.items()})
            loss_dict.update({f"model_{k}": v for k, v in loss_model.items()})
            
            # Total loss for backward
            total_loss_gen = loss_gen.pop('total_loss_for_backward')
            total_loss_model = loss_model.pop('total_loss_for_backward')
            total_loss = total_loss_gen + total_loss_model
            loss_dict['total_loss_for_backward'] = total_loss
            
            backward_context = nullcontext
            with backward_context():
                scaler.scale(total_loss).backward()
    break  # Just run one batch for testing

In [6]:
total_loss

tensor(2.0359, device='cuda:0', grad_fn=<AddBackward0>)

In [10]:
prompt_dict

{'ar_point_prompts': [None, None],
 'tc_point_prompts': [None, None],
 'ar_bbox_prompts': [None, None],
 'tc_bbox_prompts': [None, None],
 'ar_mask_prompts': [None, None],
 'tc_mask_prompts': [None, None],
 'ar_object_masks': [None, None],
 'tc_object_masks': [None, None]}